In [ ]:
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForSeq2Seq
from tqdm import tqdm

# 1. 데이터셋 및 토크나이저 설정
# 데이터세트, 토크나이저 모델 이름, 토크나이저, PAD/EOS id, 최대 길이

dataset = load_dataset("bentrevett/multi30k")
MODEL_NAME = "Helsinki-NLP/opus-mt-de-en"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
PAD_IDX = tokenizer.pad_token_id
EOS_IDX = tokenizer.eos_token_id
MAX_LENGTH = 120

def preprocess_fn(examples):
    # MarianMT 특성에 맞춰 eos_token(</s>)을 문장 앞뒤에 붙임
    src_texts = [tokenizer.eos_token + text + tokenizer.eos_token for text in examples['en']]
    tgt_texts = [tokenizer.eos_token + text + tokenizer.eos_token for text in examples['de']]

    model_input = tokenizer(
        src_texts,
        max_length=MAX_LENGTH,
        truncation=True
    )

    label = tokenizer(
        tgt_texts,
        max_length=MAX_LENGTH,
        truncation=True
    )

    model_input['labels'] = label['input_ids']
    return model_input
'''
preprocess_fn 에 들어가는 입력 예시
{
    'en': [
        "Hello world",
        "How are you?",
        "Good morning"
    ],
    'de': [
        "Hallo Welt",
        "Wie geht es dir?",
        "Guten Morgen"
    ]
}

batched = True
데이터를 한 번에 1개씩 처리하지 않고, 여러 개를 하나의 배치(Batch)로 묶어서
한 번에 처리하도록 지정하는 옵션. 해당 옵션 설정 시 batch_size = 1000이 된다.
해당 옵션 설정 시 입력으로 텍스트 리스트(List of strings)가 전달되므로
preprocess_fn 내부 로직도 리스트 입력에 맞추어 작성해야 함.
'''
# 데이터 로더에서 배치를 꺼내 출력해보면 [batch_size, 해당 데이터셋에서 가장 큰 text의 length] 차원이 나온다.
# 해당 데이터셋에서 가장 큰 text의 length 차원이 나오는 이유는 동적 패딩을 적용했기 때문.

tokenized_dataset = dataset.map(
    preprocess_fn,
    batched=True,
    remove_columns=dataset['train'].column_names
)

datacollator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    return_tensors='pt',
    label_pad_token_id=PAD_IDX
)

train_dataloader = DataLoader(
    tokenized_dataset['train'],
    batch_size=32,
    shuffle=True,
    collate_fn=datacollator
)

val_dataloader = DataLoader(
    tokenized_dataset['validation'],
    batch_size=32,
    shuffle=False,
    collate_fn=datacollator
)

# 2. 모델 클래스 정의 (batch_first=True 전용)
# 입력 / 출력 Shape: (batch_size, seq_len, emb_size)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        '''
         [
         [0], token 0
         [1], token 1
         [2], token 2
         [3], token 3
         ]
         '''
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        # [......]

        # batch_first=True 구조에 맞춰 (1, max_len, d_model) 형태로 저장
        pe = torch.zeros(1, max_len, d_model)
         '''
         [[
         [홀수 임베딩 짝수 임베딩], token 0
         [홀수 임베딩 짝수 임베딩], token 1
         [홀수 임베딩 짝수 임베딩], token 2
         [홀수 임베딩 짝수 임베딩], token 3
         ]]
         '''
        # 차원: (seq_len, d_model/2)
        pe[0, :, 0::2] = torch.sin(position * div_term).squeeze(1) # d_model이 2인 경우를 고려한 안전장치
        pe[0, :, 1::2] = torch.cos(position * div_term).squeeze(1)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x Shape: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)] # (1, x의 seq_len, d_model)
        return self.dropout(x)


# 입력 Shape: (batch_size, seq_len) -> 출력 Shape: (batch_size, seq_len, emb_size)
'''
[
[12, 123, 150, 13] #문장1
[11, 10, 1, 2, 3] #문장2
[..,,,,,,,,,,,,.] #문장3
[...,,,,,,,,,,,,] #문장4
]

>>
[
[[...], [...], [...], [...]] #문장1
[[...], [...], [...], [...]] #문장2
[..,,,,,,,,,,,,.] #문장3
[...,,,,,,,,,,,,] #문장4
]
'''
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, emb_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.emb_size = emb_size

    def forward(self, tokens):
        return self.embedding(tokens.long()) * math.sqrt(self.emb_size)


class Seq2SeqTransformer(nn.Module):
    def __init__(
        self,
        num_encoder_layers,
        num_decoder_layers,
        emb_size,
        max_len,
        nhead,
        src_vocab_size,
        tgt_vocab_size,
        dim_feedforward,
        dropout=0.1,
        pad_idx=58100
    ):
        super().__init__()
        self.pad_idx = pad_idx
        self.src_tok_emb = TokenEmbedding(src_vocab_size, emb_size)
        self.tgt_tok_emb = TokenEmbedding(tgt_vocab_size, emb_size)
        self.positional_encoding = PositionalEncoding(
            d_model=emb_size, max_len=max_len, dropout=dropout
        )

        # batch_first=True 설정 추가!
        self.transformer = nn.Transformer(
            d_model=emb_size,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation=torch.nn.functional.relu,
            layer_norm_eps=1e-05,
            batch_first=True
        )
        self.generator = nn.Linear(emb_size, tgt_vocab_size)

    def forward(
        self,
        src,                      # Shape: (batch_size, src_seq_len)
        trg,                      # Shape: (batch_size, tgt_seq_len)
        tgt_mask=None,
        src_mask=None,
        src_padding_mask=None,
        tgt_padding_mask=None,
        memory_key_padding_mask=None,
    ):
        # batch_first=True일 때 Key Padding Mask Shape: (batch_size, seq_len)
        # transpose(0, 1) 없이 바로 불리언 비교만 수행합니다.
        if src_padding_mask is None:
            src_padding_mask = (src == self.pad_idx)

        if tgt_padding_mask is None:
            tgt_padding_mask = (trg == self.pad_idx)

        if memory_key_padding_mask is None:
            memory_key_padding_mask = src_padding_mask

        # Causal Mask (미래 단어 가림 마스크) 생성
        # batch_first=True이므로 trg.size(1)이 seq_len입니다.
        if tgt_mask is None:
            tgt_len = trg.size(1)
            tgt_mask = torch.triu(
                torch.ones((tgt_len, tgt_len), device=trg.device, dtype=torch.bool),
                diagonal=1
            )

        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(trg))

        outs = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            memory_mask=None,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask,
        )
        return self.generator(outs)


# 3. 학습 준비 및 루프
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 토크나이저의 실제 vocabulary 크기를 가져옵니다.
SRC_VOCAB_SIZE = len(tokenizer)
TGT_VOCAB_SIZE = len(tokenizer)
EMB_SIZE = 512
NHEAD = 8
DIM_FEEDFORWARD = 2048
NUM_ENCODER_LAYERS = 6
NUM_DECODER_LAYERS = 6
MAX_LEN = 120
DROPOUT = 0.1
NUM_EPOCHS = 1
LEARNING_RATE = 0.0001

model = Seq2SeqTransformer(
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    emb_size=EMB_SIZE,
    max_len=MAX_LEN,
    nhead=NHEAD,
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    pad_idx=PAD_IDX
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.98), eps=1e-9)


def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for batch in tqdm(dataloader, desc="Training"):
        # DataLoader에서 나온 그대로 사용 (Shape: [batch_size, seq_len])
        src = batch['input_ids'].to(device)
        tgt = batch['labels'].to(device)

        # Teacher Forcing 분리 (1번 축이 seq_len)
        # tgt_input: (batch_size, tgt_seq_len - 1)
        # tgt_out  : (batch_size, tgt_seq_len - 1)
        tgt_input = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        optimizer.zero_grad()

        # Forward pass (transpose 없이 바로 전달)
        output = model(src, tgt_input)

        # Loss 계산을 위한 Reshape
        # output Shape : (batch_size, tgt_seq_len-1, tgt_vocab_size) -> (batch_size * (tgt_seq_len-1), tgt_vocab_size)
        # tgt_out Shape: (batch_size, tgt_seq_len-1)                -> (batch_size * (tgt_seq_len-1))
        output_dim = output.shape[-1]
        output = output.reshape(-1, output_dim)
        tgt_out = tgt_out.reshape(-1)

        loss = criterion(output, tgt_out)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) #그래디언트 클리핑
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            src = batch['input_ids'].to(device)
            tgt = batch['labels'].to(device)

            tgt_input = tgt[:, :-1]
            tgt_out = tgt[:, 1:]

            output = model(src, tgt_input)

            output_dim = output.shape[-1]
            output = output.reshape(-1, output_dim)
            tgt_out = tgt_out.reshape(-1)

            loss = criterion(output, tgt_out)
            total_loss += loss.item()

    return total_loss / len(dataloader)


# 메인 실행
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_epoch(model, train_dataloader, optimizer, criterion, device)
    valid_loss = evaluate(model, val_dataloader, criterion, device)

    print(f"[Epoch {epoch}/{NUM_EPOCHS}] Train Loss: {train_loss:.4f} | Val Loss: {valid_loss:.4f}")

torch.save(model.state_dict(), 'seq2seq_transformer.pt')

Evaluating: 100%|██████████| 32/32 [00:01<00:00, 17.52it/s]


[Epoch 1/1] Train Loss: 4.6488 | Val Loss: 3.8611


In [ ]:
class Seq2SeqTransformer(nn.Module):
    def __init__(
        self,
        num_encoder_layers,
        num_decoder_layers,
        emb_size,
        max_len,
        nhead,
        src_vocab_size,
        tgt_vocab_size,
        dim_feedforward,
        dropout=0.1,
        pad_idx=58100
    ):
        super().__init__()
        self.pad_idx = pad_idx
        self.src_tok_emb = TokenEmbedding(src_vocab_size, emb_size)
        self.tgt_tok_emb = TokenEmbedding(tgt_vocab_size, emb_size)
        self.positional_encoding = PositionalEncoding(
            d_model=emb_size, max_len=max_len, dropout=dropout
        )

        # batch_first=True 설정 추가!
        self.transformer = nn.Transformer(
            d_model=emb_size,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation=torch.nn.functional.relu,
            layer_norm_eps=1e-05,
            batch_first=True
        )
        self.generator = nn.Linear(emb_size, tgt_vocab_size)

    def forward(
        self,
        src,                      # Shape: (batch_size, src_seq_len)
        trg,                      # Shape: (batch_size, tgt_seq_len)
        tgt_mask=None,
        src_mask=None,
        src_padding_mask=None,
        tgt_padding_mask=None,
        memory_key_padding_mask=None,
    ):
        # batch_first=True일 때 Key Padding Mask Shape: (batch_size, seq_len)
        # transpose(0, 1) 없이 바로 불리언 비교만 수행합니다.
        if src_padding_mask is None:
            src_padding_mask = (src == self.pad_idx)

        if tgt_padding_mask is None:
            tgt_padding_mask = (trg == self.pad_idx)

        if memory_key_padding_mask is None:
            memory_key_padding_mask = src_padding_mask

        # Causal Mask (미래 단어 가림 마스크) 생성
        # batch_first=True이므로 trg.size(1)이 seq_len입니다.
        if tgt_mask is None:
            tgt_len = trg.size(1)
            tgt_mask = torch.triu(
                torch.ones((tgt_len, tgt_len), device=trg.device, dtype=torch.bool),
                diagonal=1
            )

        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(trg))

        outs = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            memory_mask=None,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask,
        )
        return self.generator(outs)

    def encode(self, src, src_mask=None):
        """인코더 전용 연산: 입력 문장을 컨텍스트 표현(memory)으로 변환"""
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        return self.transformer.encoder(src_emb, mask=src_mask)

    def decode(self, tgt, memory, tgt_mask=None):
        """디코더 전용 연산: 이전까지 생성된 토큰들과 인코더 표현(memory)을 받아 다음 토큰 생성"""
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt))
        return self.transformer.decoder(tgt_emb, memory, tgt_mask=tgt_mask)

In [ ]:
model = Seq2SeqTransformer(
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    emb_size=EMB_SIZE,
    max_len=MAX_LEN,
    nhead=NHEAD,
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    pad_idx=PAD_IDX
).to(device)
model.load_state_dict(torch.load('seq2seq_transformer.pt', map_location=device))
model.eval()  # Evaluation 모드로 전환 (Dropout 비활성화)
EOS_IDX = tokenizer.eos_token_id


# 4. Greedy Search 디코딩 함수
def greedy_decode(model, src_sentence, max_len=120):
    model.eval()

    # 입력 문장 토크나이징 (전처리와 동일하게 </s>로 시작/종료)
    formatted_src = tokenizer.eos_token + src_sentence + tokenizer.eos_token
    src_tokens = tokenizer.encode(formatted_src, return_tensors='pt').to(device) # Shape: (1, src_seq_len)

    with torch.no_grad():
        # Step 1: Encoder 출력(memory) 계산 (한 번만 수행)
        memory = model.encode(src_tokens)

        # Step 2: Decoder 시작 토큰(EOS/SOS)으로 시작 텐서 생성
        ys = torch.ones(1, 1).fill_(EOS_IDX).type(torch.long).to(device) # Shape: (1, 1)

        # Step 3: 한 토큰씩 예측 반복
        for _ in range(max_len - 1):
            memory = memory.to(device)

            # Causal Mask (미래 단어 가림 마스크) 생성
            tgt_len = ys.size(1)
            tgt_mask = torch.triu(
                torch.ones((tgt_len, tgt_len), device=device, dtype=torch.bool),
                diagonal=1
            )

            # Decoder 계산
            out = model.decode(ys, memory, tgt_mask)
            prob = model.generator(out[:, -1])  # 가장 마지막 시점의 토큰 확률만 추출
            _, next_word = torch.max(prob, dim=1) # 가장 높은 확률의 토큰 선택
            next_word = next_word.item()

            # 생성된 토큰을 기존 출력 sequence 뒤에 붙임
            ys = torch.cat([ys, torch.ones(1, 1).type_as(src_tokens.data).fill_(next_word)], dim=1)

            # EOS(</s>) 토큰이 나오면 생성 종료
            if next_word == EOS_IDX:
                break

    # 토큰 ID 배열을 다시 문장으로 디코딩
    translated_tokens = ys.squeeze(0).tolist()
    translated_text = tokenizer.decode(translated_tokens, skip_special_tokens=True)
    return translated_text


# 5. 번역 테스트 실행
sample_sentences = [
    "A dog is running in the park.",
    "A man is playing a guitar on the street.",
    "Two children are playing with a ball."
]

for sentence in sample_sentences:
    translated = greedy_decode(model, sentence, max_len=MAX_LEN)
    print(f"Input  (EN): {sentence}")
    print(f"Output (DE): {translated}")
    print("-" * 50)

Input  (EN): A dog is running in the park.
Output (DE): Ein Hund springt im Schnee.
--------------------------------------------------
Input  (EN): A man is playing a guitar on the street.
Output (DE): Ein Mann spielt auf einem Fahrrad.
--------------------------------------------------
Input  (EN): Two children are playing with a ball.
Output (DE): Zwei Kinder spielen mit einem Feld.
--------------------------------------------------
